# Customer Lifetime Value Prediction

**Tabular Regression Project**

Models: CatBoost · LightGBM · XGBoost
Baselines: LazyPredict · FLAML AutoML
Target: `clv`

## 2 · Project Overview

We predict **Customer Lifetime Value (CLV)** based on purchase behavior (spend, frequency, order value), customer tenure, segment, channel, returns, support interactions, satisfaction, and email engagement.

## 3 · Learning Objectives

By the end of this notebook you will be able to:
1. Perform EDA on a tabular regression dataset.
2. Encode categorical features for tree-based and linear models.
3. Build a baseline Linear Regression model.
4. Use LazyPredict for rapid model benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, XGBoost with GPU→CPU fallback.
7. Evaluate with RMSE, MAE, R², and residual analysis.

## 4 · Problem Statement

Given a customer's transaction history, engagement metrics, and demographic segment, predict their total lifetime value to the business.

## 5 · Why This Project Matters

- **CLV prediction** drives customer acquisition budgets and retention investment.
- Understanding high-value customer traits enables targeted marketing.
- Churn prevention ROI depends on accurate CLV estimates.
- Demonstrates regression with customer-centric business features.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 7,500 |
| **Features** | tenure_months, avg_monthly_spend, purchase_frequency, avg_order_value, num_returns, customer_segment, channel, support_tickets, satisfaction_score, email_engagement |
| **Target** | clv (continuous, USD) |
| **Range** | ~$50 – $100,000 |

## 7 · Dataset Source and License Notes

- **Source**: Synthetic dataset generated for educational purposes.
- **License**: Public / educational use. No PII.
- **Limitations**: Simplified patterns — real-world data would have more features and noise.

## 8 · Environment Setup

Install any missing packages. GPU is auto-detected with CPU fallback.

In [ ]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)

print("All packages ready.")

## 9 · Imports

In [ ]:
import os, json, time, warnings, random, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, root_mean_squared_error

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

## 10 · Configuration / Constants

In [ ]:
TARGET = "clv"
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
print(f"Target: {TARGET}")
print(f"Save dir: {SAVE_DIR}")

## 11 · Dataset Download or Loading

Synthetic dataset: 7,500 customer records with purchase history and engagement features.

In [ ]:
np.random.seed(SEED)
n = 7500
tenure_months = np.random.randint(1, 120, n)
avg_monthly_spend = np.round(np.random.lognormal(3.5, 0.8, n).clip(5, 2000), 2)
purchase_frequency = np.round(np.random.exponential(3, n).clip(0.5, 30), 1)
avg_order_value = np.round(np.random.lognormal(3.2, 0.6, n).clip(10, 800), 2)
num_returns = np.random.poisson(1.5, n).clip(0, 15)
customer_segment = np.random.choice(["Budget", "Regular", "Premium", "VIP"], n,
                                     p=[0.3, 0.35, 0.25, 0.1])
seg_mult = {"Budget": 0.7, "Regular": 1.0, "Premium": 1.5, "VIP": 2.5}
seg_val = np.array([seg_mult[s] for s in customer_segment])

channel = np.random.choice(["Online", "In-Store", "Both"], n, p=[0.35, 0.3, 0.35])
chan_mult = {"Online": 0.9, "In-Store": 1.0, "Both": 1.2}
chan_val = np.array([chan_mult[c] for c in channel])

support_tickets = np.random.poisson(2, n).clip(0, 20)
satisfaction_score = np.round(np.random.normal(3.8, 0.7, n).clip(1, 5), 1)
email_engagement = np.round(np.random.beta(3, 5, n), 2)

# CLV model: cumulative spending + retention factor
retention_factor = np.minimum(tenure_months / 12, 8)
clv = (avg_monthly_spend * purchase_frequency * retention_factor
       + avg_order_value * 2
       - 50 * num_returns
       - 30 * support_tickets
       + 200 * satisfaction_score
       + 500 * email_engagement)
clv = clv * seg_val * chan_val
clv = np.round(clv + np.random.normal(0, 300, n), 2).clip(50, 100000)

df = pd.DataFrame({
    "tenure_months": tenure_months, "avg_monthly_spend": avg_monthly_spend,
    "purchase_frequency": purchase_frequency, "avg_order_value": avg_order_value,
    "num_returns": num_returns, "customer_segment": customer_segment,
    "channel": channel, "support_tickets": support_tickets,
    "satisfaction_score": satisfaction_score, "email_engagement": email_engagement,
    "clv": clv
})
print(f"Dataset shape: {df.shape}")
print(f"Target stats:\n{df['clv'].describe()}")
df.head()

## 12 · Data Validation Checks

Verify the dataset is complete and correctly structured.

In [ ]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed.")
print(f"\nTarget stats:\n{df[TARGET].describe()}")

## 13 · Exploratory Data Analysis

Explore feature distributions, correlations, and relationships.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

axes[0][0].scatter(df["tenure_months"], df[TARGET], alpha=0.2, s=10)
axes[0][0].set_xlabel("Tenure (months)"); axes[0][0].set_ylabel("CLV")
axes[0][0].set_title("Tenure vs CLV")

axes[0][1].scatter(df["avg_monthly_spend"], df[TARGET], alpha=0.2, s=10)
axes[0][1].set_xlabel("Avg Monthly Spend"); axes[0][1].set_ylabel("CLV")
axes[0][1].set_title("Monthly Spend vs CLV")

axes[0][2].scatter(df["purchase_frequency"], df[TARGET], alpha=0.2, s=10)
axes[0][2].set_xlabel("Purchase Frequency"); axes[0][2].set_ylabel("CLV")
axes[0][2].set_title("Frequency vs CLV")

df.boxplot(column=TARGET, by="customer_segment", ax=axes[1][0])
axes[1][0].set_title("CLV by Customer Segment")

df.boxplot(column=TARGET, by="channel", ax=axes[1][1])
axes[1][1].set_title("CLV by Channel")

corr = df.select_dtypes(include="number").corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=axes[1][2], square=True, cbar_kws={"shrink": 0.8})
axes[1][2].set_title("Correlation")

plt.tight_layout()
plt.show()

## 14 · Target Analysis

Examine the distribution of `clv`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df[TARGET], bins=30, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].set_title(f"Distribution of {TARGET}")
axes[0].set_xlabel("Customer Lifetime Value ($)")

df.boxplot(column=TARGET, by="customer_segment", ax=axes[1])
axes[1].set_title("CLV by Segment")

plt.tight_layout()
plt.show()
print(f"Range: ${df[TARGET].min():,.0f} to ${df[TARGET].max():,.0f}")
print(f"Mean: ${df[TARGET].mean():,.0f}, Median: ${df[TARGET].median():,.0f}")
print(f"Skew: {df[TARGET].skew():.2f}")

## 15 · Train/Validation/Test Split Strategy

80/20 train/test split.

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Encode categorical features
cat_cols = X.select_dtypes(include="object").columns.tolist()
if cat_cols:
    oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    X[cat_cols] = oe.fit_transform(X[cat_cols])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Target — Train mean: {y_train.mean():,.2f}, Test mean: {y_test.mean():,.2f}")

## 16 · Preprocessing Strategy

- **Missing values**: None.
- **Categorical**: `customer_segment`, `channel` encoded with OrdinalEncoder.
- **Scaling**: Not needed for tree models.
- **Outliers**: VIP CLV values are genuine high-value customers.

## 17 · Feature Engineering

Sanitize column names and add engineered features.

In [ ]:
X_train = X_train.copy()
X_test = X_test.copy()

X_train["spend_x_frequency"] = X_train["avg_monthly_spend"] * X_train["purchase_frequency"]
X_test["spend_x_frequency"] = X_test["avg_monthly_spend"] * X_test["purchase_frequency"]

X_train["retention_years"] = np.minimum(X_train["tenure_months"] / 12, 8)
X_test["retention_years"] = np.minimum(X_test["tenure_months"] / 12, 8)

X_train["net_satisfaction"] = X_train["satisfaction_score"] - 0.3 * X_train["support_tickets"]
X_test["net_satisfaction"] = X_test["satisfaction_score"] - 0.3 * X_test["support_tickets"]

clean = [c.replace(" ", "_").replace("-", "_").replace(".", "_") for c in X_train.columns]
X_train.columns = clean
X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

## 18 · Baseline Model

Linear Regression as our baseline regressor.

In [ ]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)

rmse_bl = root_mean_squared_error(y_test, y_pred_bl)
mae_bl = mean_absolute_error(y_test, y_pred_bl)
r2_bl = r2_score(y_test, y_pred_bl)

print("=" * 50)
print("BASELINE: Linear Regression")
print("=" * 50)
print(f"RMSE : {rmse_bl:,.2f}")
print(f"MAE  : {mae_bl:,.2f}")
print(f"R2   : {r2_bl:.4f}")

## 19 · LazyPredict Benchmark

Fit dozens of regressors in one call for a quick ranking.

In [ ]:
from lazypredict.Supervised import LazyRegressor

lazy = LazyRegressor(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)

print("LazyPredict — Top 15 Regressors:")
print(lazy_models.head(15).to_string())

## 20 · FLAML AutoML Run

Automated model selection and hyperparameter tuning (60s budget).

In [ ]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(X_train, y_train, task="regression", time_budget=60,
                 estimator_list=["lgbm", "rf", "extra_tree", "catboost"],
                 verbose=0, seed=SEED)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best model: {flaml_automl.best_estimator}")
print(f"RMSE: {root_mean_squared_error(y_test, y_pred_flaml):,.2f}")
print(f"R2  : {r2_score(y_test, y_pred_flaml):.4f}")

## 21 · Additional Models: CatBoost, LightGBM, XGBoost

Train the modern tabular model stack. GPU auto-detected with CPU fallback.

In [ ]:
def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}
timings = {}

# CatBoost
try:
    from catboost import CatBoostRegressor
    t0 = time.perf_counter()
    try:
        cb = CatBoostRegressor(iterations=300, learning_rate=0.05, depth=6,
                                task_type="GPU", devices="0", verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    except Exception:
        cb = CatBoostRegressor(iterations=300, learning_rate=0.05, depth=6,
                                verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    timings["CatBoost"] = time.perf_counter() - t0
    results["CatBoost"] = cb.predict(X_test)
    print(f"CatBoost RMSE: {root_mean_squared_error(y_test, results['CatBoost']):,.2f}  ({timings['CatBoost']:.1f}s)")
except Exception as e:
    print(f"CatBoost failed: {e}")
gpu_cleanup()

# LightGBM
try:
    import lightgbm as lgb
    t0 = time.perf_counter()
    try:
        lg = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                device="gpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    except Exception:
        lg = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    timings["LightGBM"] = time.perf_counter() - t0
    results["LightGBM"] = lg.predict(X_test)
    print(f"LightGBM RMSE: {root_mean_squared_error(y_test, results['LightGBM']):,.2f}  ({timings['LightGBM']:.1f}s)")
except Exception as e:
    print(f"LightGBM failed: {e}")
gpu_cleanup()

# XGBoost
try:
    from xgboost import XGBRegressor
    t0 = time.perf_counter()
    try:
        xgb_model = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  device="cuda", tree_method="hist", verbosity=0,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    except Exception:
        xgb_model = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  tree_method="hist", verbosity=0,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    timings["XGBoost"] = time.perf_counter() - t0
    results["XGBoost"] = xgb_model.predict(X_test)
    print(f"XGBoost RMSE: {root_mean_squared_error(y_test, results['XGBoost']):,.2f}  ({timings['XGBoost']:.1f}s)")
except Exception as e:
    print(f"XGBoost failed: {e}")
gpu_cleanup()

# Add baseline & FLAML
results["Linear Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

## 22 · Top 3 Model Selection

Rank all models by RMSE and select the top 3.

In [ ]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "RMSE": round(root_mean_squared_error(y_test, yp), 2),
        "MAE": round(mean_absolute_error(y_test, yp), 2),
        "R2": round(r2_score(y_test, yp), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("RMSE")
print("All Model Rankings (by RMSE):")
print(scores_df.to_string())

top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3 models: {top3_names}")

## 23 · Final Training and Evaluation of Top 3

Detailed metrics and predicted-vs-actual plots.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    rmse = root_mean_squared_error(y_test, yp)
    r2 = r2_score(y_test, yp)

    axes[i].scatter(y_test, yp, alpha=0.6, s=40, edgecolors="black")
    mn, mx = min(y_test.min(), yp.min()), max(y_test.max(), yp.max())
    axes[i].plot([mn, mx], [mn, mx], "r--", lw=2)
    axes[i].set_title(f"{name}\nRMSE={rmse:,.2f}  R2={r2:.4f}")
    axes[i].set_xlabel("Actual")
    axes[i].set_ylabel("Predicted")

plt.suptitle("Top 3 Models — Predicted vs Actual", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "top3_predicted_vs_actual.png"), dpi=120, bbox_inches="tight")
plt.show()

print("\nDetailed Metrics — Top 3:")
for name in top3_names:
    yp = results[name]
    print(f"\n  {name}:")
    print(f"    RMSE : {root_mean_squared_error(y_test, yp):,.2f}")
    print(f"    MAE  : {mean_absolute_error(y_test, yp):,.2f}")
    print(f"    R2   : {r2_score(y_test, yp):.4f}")

## 24 · Error Analysis

Examine residuals from the best model.

In [ ]:
best_name = top3_names[0]
best_pred = results[best_name]
residuals = y_test.values - best_pred

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(residuals, bins=20, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].axvline(0, color="red", linestyle="--")
axes[0].set_title(f"{best_name} — Residual Distribution")
axes[0].set_xlabel("Residual (Actual - Predicted)")

axes[1].scatter(best_pred, residuals, alpha=0.6, s=40, edgecolors="black")
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_title(f"{best_name} — Residual vs Predicted")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Residual")

sorted_idx = np.argsort(best_pred)
axes[2].scatter(best_pred[sorted_idx], np.abs(residuals[sorted_idx]), alpha=0.6, s=40, edgecolors="black")
axes[2].set_title(f"{best_name} — |Residual| vs Predicted")
axes[2].set_xlabel("Predicted"); axes[2].set_ylabel("|Residual|")

plt.tight_layout()
plt.show()

print(f"Residual stats ({best_name}):")
print(f"  Mean: {residuals.mean():,.2f}, Std: {residuals.std():,.2f}")
print(f"  Min: {residuals.min():,.2f}, Max: {residuals.max():,.2f}")
print(f"  Median: {np.median(residuals):,.2f}")

## 25 · Interpretation and Business Insight

**Key findings:**
- **Spend × frequency** is the dominant CLV driver (monetary velocity).
- **Customer segment** (VIP: 2.5× vs Budget: 0.7×) sets the value tier.
- **Tenure** drives CLV through retention factor (capped at ~8 years).
- **Returns and support tickets** reduce CLV measurably.
- **Omnichannel customers** (Both) have 20% higher CLV.

**Business takeaway:** Invest in converting Regular→Premium customers and reducing returns. Omnichannel engagement is the strongest loyalty signal.

## 26 · Limitations

1. Synthetic data with simplified CLV calculation.
2. No discount rate for future value.
3. Missing churn probability component.
4. No product category breakdown.
5. Simplified segment definitions.

## 27 · How to Improve This Project

1. Use real transaction data with probabilistic CLV models (BG/NBD).
2. Add churn probability as input feature.
3. Include product category-level purchase behavior.
4. Apply discount rate for time-value of money.
5. Build cohort-based CLV models.

## 28 · Production Considerations

- Deploy for customer scoring and segmentation.
- Trigger retention campaigns for high-CLV at-risk customers.
- Size acquisition budgets by expected CLV.
- Monitor CLV drift over time.
- Output CLV distributions, not just point estimates.

## 29 · Common Mistakes

1. Not accounting for tenure in CLV (short-tenure high-spenders may churn).
2. Ignoring returns as a negative CLV signal.
3. Treating all segments equally in marketing spend.
4. Using simple averages instead of probabilistic CLV.
5. Not capping tenure contribution (diminishing retention returns).

## 30 · Mini Challenge / Exercises

1. Log-transform CLV — does R² improve for Linear Regression?
2. Remove `customer_segment` — how much does RMSE increase?
3. Create `return_rate = num_returns / purchase_frequency` and test.
4. Build separate models for Online vs. In-Store vs. Both.
5. Plot CLV vs. tenure — confirm the plateau effect.

## 31 · Final Summary / Key Takeaways

1. **Monetary velocity** (spend × frequency) is the primary CLV driver.
2. **Customer segment** multiplies value by 0.7× to 2.5×.
3. **Tenure** contributes through retention, capped around 8 years.
4. **Returns and support issues** reduce CLV measurably.
5. **Omnichannel customers** are the most valuable.
6. **CLV prediction** enables data-driven marketing budget allocation.

## Save Metrics

In [ ]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "rmse": round(float(root_mean_squared_error(y_test, yp)), 2),
        "mae": round(float(mean_absolute_error(y_test, yp)), 2),
        "r2": round(float(r2_score(y_test, yp)), 4),
    }

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))